# cleaning.ipynb

Combine and reformat raw data for use in modeling further in the pipeline.

In [29]:
# Globals for this file, need to be updated once
# a new season, or sport is added
SPORTS_SEASONS = {"nfl":[2025], 
                  "mlb":[2025], 
                  "nba":[2025], 
                  "nhl":[2025]}

STATES = ["away", "home"]

import csv
import sys
import pandas as pd

sys.path.append("..")
from src.pricing.devig import devig_two_way, american_odds_to_prob

In [34]:
def read_season_data_file(sport: str, season: int) -> pd.DataFrame:
    """Read in .csv file from data directory and return dataframe containing the data"""
    if sport not in SPORTS_SEASONS.keys() or season not in SPORTS_SEASONS[sport]:
        raise ValueError("Dataset not currently available.")

    df = pd.read_csv(f"../data/{sport}_{season}_historical.csv")

    return df

def clean_data(df: pd.DataFrame, de_vig_strat: str = "cheung") -> pd.DataFrame:
     """Clean datasets for analysis"""
     # Drop useless columns
     df = df.drop(columns=["game_id",
                      "kalshi_home_ticker",
                      "kalshi_away_ticker", 
                      "kalshi_home_candle_end_ts", 
                      "kalshi_away_candle_end_ts",
                      "sportsbook_bookmaker"])

     df = df.dropna()

     df["total_volume"] = df[f"kalshi_{STATES[0]}_volume"] + df[f"kalshi_{STATES[1]}_volume"]

    # Compute Kalshi implied prob
     for state in STATES:
          df[f"Kalshi_{state}_prob"] = (df[f"kalshi_{state}_yes_bid_close"] + df[f"kalshi_{state}_yes_ask_close"]) / 2
          df = df.drop(columns=[f"kalshi_{state}_close_price",
                                f"kalshi_{state}_yes_bid_close",
                                f"kalshi_{state}_yes_ask_close"])

     
     df[[f"sportsbook_{STATES[0]}_prob", f"sportsbook_{STATES[1]}_prob"]] = df.apply(
          lambda row: devig_two_way(
               american_odds_to_prob(row[f"sportsbook_{STATES[0]}_odds_american"]),
               american_odds_to_prob(row[f"sportsbook_{STATES[1]}_odds_american"]),
               de_vig_strat=de_vig_strat,
          ),
          axis=1,
          result_type="expand",
     )

     df = df.drop(columns=[f"sportsbook_{STATES[0]}_odds_american",
                           f"sportsbook_{STATES[1]}_odds_american",
                           ])

     for state in STATES:
          df[f"{state}_prob_spread"] = df[f"sportsbook_{state}_prob"] - df[f"Kalshi_{state}_prob"]

     return df

def combine_data(dfs: list[pd.DataFrame]) -> pd.DataFrame:
     """Combine specified dataframes into one dataframe (chronological order)"""
     pass

In [36]:
tmp = read_season_data_file("nfl", 2025)
tmp = clean_data(tmp)

In [38]:
# Read and clean all sports for 2025 season
raw = {}
clean = {}

for sport in SPORTS_SEASONS.keys():
    raw[sport] = read_season_data_file(sport, 2025)
    clean[sport] = clean_data(raw[sport])
    display(clean[sport].head())

,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_away_volume,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
1,New England Patriots,Houston Texans,2026-01-18T20:00Z,28,16,11354.0,35912.0,47266.0,0.375,0.635,0.378989,0.621011,0.003989,-0.013989
3,Denver Broncos,Buffalo Bills,2026-01-17T21:30Z,33,30,99259.0,135805.0,235064.0,0.465,0.535,0.466061,0.533939,0.001061,-0.001061
4,Chicago Bears,Los Angeles Rams,2026-01-18T23:30Z,17,20,189690.0,102558.0,292248.0,0.645,0.355,0.650625,0.349375,0.005625,-0.005625
5,Chicago Bears,Green Bay Packers,2026-01-11T01:00Z,31,27,138420.0,71119.0,209539.0,0.535,0.465,0.544591,0.455409,0.009591,-0.009591
6,Carolina Panthers,Los Angeles Rams,2026-01-10T21:30Z,31,34,41533.0,23321.0,64854.0,0.815,0.185,0.833306,0.166694,0.018306,-0.018306


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_away_volume,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,Toronto Blue Jays,Los Angeles Dodgers,2025-11-01T00:00Z,1.0,3.0,9526.0,101107.0,110633.0,0.565,0.435,0.565363,0.434637,0.000363,-0.000363
1,Los Angeles Dodgers,Toronto Blue Jays,2025-10-30T00:00Z,1.0,6.0,54742.0,5664.0,60406.0,0.355,0.645,0.349375,0.650625,-0.005625,0.005625
2,Los Angeles Dodgers,Toronto Blue Jays,2025-10-29T00:00Z,2.0,6.0,12994.0,19302.0,32296.0,0.355,0.645,0.349375,0.650625,-0.005625,0.005625
3,Los Angeles Dodgers,Toronto Blue Jays,2025-10-28T00:00Z,6.0,5.0,13586.0,3276.0,16862.0,0.355,0.645,0.355039,0.644961,0.000039,-0.000039
4,Toronto Blue Jays,Los Angeles Dodgers,2025-10-26T00:00Z,1.0,5.0,6287.0,5819.0,12106.0,0.555,0.440,0.565363,0.434637,0.010363,-0.005363


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_away_volume,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,San Antonio Spurs,New York Knicks,2026-06-14T00:30Z,90.0,94.0,138527.57,268615.60,407143.17,0.365,0.645,0.355039,0.644961,-0.009961,-0.000039
1,New York Knicks,San Antonio Spurs,2026-06-11T00:30Z,107.0,106.0,148522.77,152017.60,300540.37,0.465,0.545,0.445237,0.554763,-0.019763,0.009763
2,New York Knicks,San Antonio Spurs,2026-06-09T00:30Z,111.0,115.0,98582.65,188022.49,286605.14,0.475,0.525,0.466061,0.533939,-0.008939,0.008939
3,San Antonio Spurs,New York Knicks,2026-06-06T00:30Z,104.0,105.0,35197.11,166617.94,201815.05,0.315,0.685,0.317028,0.682972,0.002028,-0.002028
4,San Antonio Spurs,New York Knicks,2026-06-04T00:30Z,95.0,105.0,59303.66,144635.50,203939.16,0.365,0.635,0.355039,0.644961,-0.009961,0.009961


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_away_volume,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,Vegas Golden Knights,Carolina Hurricanes,2026-06-15T00:00Z,0.0,3.0,51541.43,13370.39,64911.82,0.525,0.485,0.511370,0.488630,-0.013630,0.003630
1,Carolina Hurricanes,Vegas Golden Knights,2026-06-12T00:00Z,4.0,2.0,8496.86,63315.01,71811.87,0.405,0.595,0.414092,0.585908,0.009092,-0.009092
2,Vegas Golden Knights,Carolina Hurricanes,2026-06-10T00:00Z,3.0,5.0,18936.48,14659.86,33596.34,0.515,0.485,0.511370,0.488630,-0.003630,0.003630
3,Vegas Golden Knights,Carolina Hurricanes,2026-06-07T00:00Z,5.0,4.0,70448.68,29760.76,100209.44,0.505,0.495,0.500000,0.500000,-0.005000,0.005000
4,Carolina Hurricanes,Vegas Golden Knights,2026-06-05T00:00Z,4.0,3.0,24634.37,15634.68,40269.05,0.395,0.605,0.405664,0.594336,0.010664,-0.010664
